# Jupyter and Pandas: First Steps
Pandas is a popular python tool for managing and analyzing tabular data.  It is extremely useful for working through big data and doing quick statistics on them.

Notebook outline:

* Import libraries
* Load data from disk
* Inspect data and transform to make it more convenient to work with

## First Steps
Start a jupyter notebook: `jupyter notebook`

In [ ]:
import pandas as pd
import numpy as np

Creating a Series.  _A pandas Series is a basic data structure, and can be thought of like a column in a spreadsheet._

In [ ]:
points = pd.Series([0, 5, 10, 3])
ages = pd.Series([18, 22, 25, 30])

Creating a Data Frame.  _A DataFrame is the other major data structure in pandas.  It can be thought of like a full spreadsheet, with key value pairs denoting the label, and values of the entries._

In [ ]:
gamedata = pd.DataFrame({
    "points": points,
    "player":["Jim", "Joe", "Bob", "Tracy"],
    "ages": ages
})

### Load Data
Pandas provides a set of methods that can be used to load data. In general, these start with `pd.read_`. The methods including reading csv files `read_csv`, JSON (`read_json`), Excel (`read_excel`), and many others. In this activity, we will be using data taken from the Framingham Heart Study. The Framingham study is one of the most famous studies in medical literature and covers a large set of patients with decades worth of outcome and follow-up.

Load a CSV file (fetches data from a remote URL):

In [ ]:
patientdata = pd.read_csv('https://oak-tree.tech/documents/96/framingham.csv')

### Inspecting the Data
Pandas makes it easy to inspect the contents of a data file. It provides methods and properties for introspecting the columns and data types. _Looking at how data is imported is a good way to get your bearings. The `head` method lets you see the first few rows of data. It also accepts an argument which allows you to specify how many rows you want._

In [ ]:
patientdata.head()

In [ ]:
# View more than the first 5 rows by passing in an explicit 
# value to head
patientdata.head(10)

Retrieve information about the data columns in the frame.

In [ ]:
# The columns are an instance of Index
patientdata.columns

`dtype` reveals the full list of columns, and their types:

In [ ]:
# When looking at the data, ensure that the columns have a correct type (object could mean string)
patientdata.dtypes

`info` provides information about the size of the data on disk, as well as how many non-null entries each column has:

In [ ]:
# this is helpful in knowing how much space the data takes up
patientdata.info()

You can transpose a DataFrame with either `.T` or `.transpose()`. _This often makes it easier to view._

In [ ]:
patientdata.transpose() # Alternative: patientdata.transpose()

`shape` returns the size of the data (number of row, number of columns):

In [ ]:
patientdata.shape

Index is what Pandas uses to iterate through rows:

In [ ]:
patientdata.index

It can be set to a different column. It can be set to a different column. _Code below creates a random number associated with the rows, which is then set to `RANDID`._

In [ ]:
patientdata['RANDID'] = np.random.choice(len(patientdata), len(patientdata), replace=False)
patientdata.set_index('RANDID')

You can revert a `.set_index` with a `.reset_index`:

In [ ]:
patientdata.set_index('RANDID').reset_index()

### Preparing Data for Analysis
The Framingham heart data is a very clean dataset. It's been maintained for more than fifty years and is widely used by many institutions.  Most of the time this is not the case.

> In Data Science, 80% of time is spent preparing data, 20% of time spent complaining about the need to prepare data.

Let's see how 80% of the time is spent.

Start by inspecting the columns. Pandas tries to do the right thing and infer type, but that doesn't always happen correctly. _Frequently data is a mess._

In [ ]:
patientdata.dtypes

Start by removing whitespace from in front of volume `Columns`. _Names that comply with Python naming rules allow  you to access the column as a property of the DataFrame._

In [ ]:
# The structure used here for the iteration is called a list comprehension in Python.
# They provide a convenient way to perform iterations and return the result as a list.
patientdata.colums = [c.strip() for c in patientdata.columns]

# Replace spaces with underscorees (comply with Python attribute naming rules)
patientdata.colums = [c.replace(' ', '_') for c in patientdata.columns]

Mixed data types makes it impossible to aggregate data. The BMI column contains entries of `abstain` for people that didn't want to be weighed. It is possible to change that to `NaN` (not a number) by replacing a string value and then converting that to a numeric value:

In [ ]:
patientdata.BMI.replace('abstain', '')

Pandas contains a large set of convenience functions that can be used to coerce data to a specific format. For example, `to_numeric` which can be used to convert column elements to numeric values:

In [ ]:
patientdata.BMI = pd.to_numeric(patientdata.BMI)

If you wish to provide default value instead of NaN, you can change it for a whole column at once.

_Here we have chosen to use 0 for RANDID's NaN because no id is currently set at 0, and it will collect them all into an anonymous user if we want to sort by id and make sure we get everyone.  When working with data there is no hard and fast rule about recording or removing values from analysis.  You need to make the right decision for your data based on the type of analysis you will be performing._

Other convenient properties and methods:

* If you wish to modify `str` data (for example, convert to upper or lowercase), it is possible to execute operations on string columns using the `str` attribute. Example: `data.col.str.uppser()`.

### Re-Mapping Data
New columns can be added to an existing dataset by specifying a column name and providing values.

In [ ]:
# Create a column to hold a string representation of whether the patient died.
# Setting it to '' uses that for all rows in the dataset.
patientdata['CHDSTR'] = ''

# Re-code the patient's age in days
patientdata['AGEDAYS'] = patientdata.age * 365

It is possible to recode data to a new column using an arbitrary function. The general procedure:

* copy old data to the new column
* create a function to handle the recoding
* apply the operation

In [ ]:
# Arbitrary transform functions are slow as they are not vectorized.
# There are several methods that each work somewhat differently.
# map - works with a dictionary (mapping value to new value), series (like dict), function
# apply - only works with function as a parameter. Allows extra parameters.
# aggregate (agg) works with function or list of functions. If reducing function, returns a scalar.
# transform - wraps agg and applies an operation in place
def bool2str(value):
    if not value:
        return "False"
    return "True"

# Create new column with a blank string
patientdata.CHD = ''

# Copy the CHD data to the new column
patientdata.CHD = patientdata.TenYearCHD

# Map the data to the new column (in place)
patientdata['CHD'] = patientdata.CHD.transform(bool2str)

# Assign to new column name
patientdata['CHD_DUP'] = patientdata.CHD.map(bool2str)

_In Framingham dataset, yes/no variables are generally represented as 0 (no) and 1 (yes). Categorical variables have been coded as integers. These conventions allow for the processing library to process operations efficiently._

Return a count of how frequently different values appear.

In [ ]:
patientdata.CHD.value_counts()

Remove the new column.  _We use `axis=1` to tell it we're dropping a column._

In [ ]:
patientdata = patientdata.drop(['CHDSTR'], axis=1)

_Pandas provides a broad range of features for working with data. We will look at additional examples throughout the remainder of this chapter. This includes mechanisms for working with data from SQL databases._